# F02-P2 Climate

**Site Characterization: Climate, components 3.x.**

Quantifies the carbon side of the site: what is stored now, and later what is at stake and what
could be gained. Where General Context and Nature describe the site, Climate puts numbers on the
climate value of it.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** 3.1 Current Carbon Storage is written. Later components are not started.

## Handoff

Reads nothing from earlier stages at present. Writes
`outputs/<aoi_id>__F02-P2-climate.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass, replace

import geopandas as gpd
import numpy as np

from config import *
from common import *

In [ ]:
AOI_PATH = r"<SET: path to the project AOI polygon>"
aoi_id = "<SET: short id for this run, must match the other F02-P2 notebooks>"

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

---
## 3.1 Current Carbon Storage

Reports how much carbon the project area holds right now, as a single headline number in tonnes
of CO2 equivalent, with a breakdown per carbon pool in tCO2e and in percent.

**Data.** Two continuous rasters holding **dry biomass density in Mg/ha**, not carbon:

- `agb_mgha.tif` - aboveground biomass. The in-house layer, Alpha Earth plus GEDI.
- `bgb_mgha.tif` - belowground biomass, that is roots.

**What this number covers, and what it does not.** Two of the five IPCC carbon pools are
included: aboveground biomass and belowground biomass. Deadwood, litter and **soil organic
carbon are all excluded**.

The soil exclusion matters most on peat. In tropical peat swamp forest the biomass pools
typically hold only a small share of total site carbon, often in the region of 5 to 15 percent,
because the peat itself can store on the order of thousands of tCO2e per hectare depending on
depth. A peatland AOI will therefore report a headline that is far below its real stock. Since
Peatland is one of the three Axis 3 classes in 1.1, this case will occur. The component
therefore labels its output as biomass carbon rather than total carbon, and carries the pool
list in `values` so the frontend can state the scope next to the big number.

**Read the pool split with care.** If `bgb_mgha.tif` was produced upstream by multiplying AGB by
a fixed root to shoot ratio, then the AGB and BGB shares are constant by construction: a ratio
of 0.25 always yields an 80 / 20 split, on every site. The percentages would then carry no site
specific information and should not be presented as a finding. The split is only informative if
BGB was mapped independently, or if the root to shoot ratio varies by ecosystem or biomass
class. This needs one check against how the BGB layer was built.

**The two conversions, kept visible in the tool.**

```
carbon_tC    = biomass_Mg * CARBON_FRACTION      # 0.47, IPCC 2006 GL Vol 4 Ch 4
storage_tCO2e = carbon_tC * CO2_PER_C            # 44 / 12 = 3.667
```

Neither step is done upstream. Reading the biomass rasters as biomass and applying both factors
here means the assumptions live in `config.py` where they can be audited and changed, rather
than being baked into a raster that looks like a measurement.

**Decisions locked.**

- Extent is every valid pixel of the biomass rasters inside the AOI, not forest only. The
  headline claims the stock of the site, so shrubland, agroforestry and plantation biomass count.
- Nodata inside the AOI is treated as zero biomass. Accepted by the team. The component still
  measures raster coverage and flags an AOI where coverage falls below
  `CARBON_COVERAGE_WARN_PCT`, because with nodata read as zero an incomplete raster produces a
  quiet under-estimate that the number itself cannot reveal.
- The two pools are integrated separately and then added, so AGB and BGB may sit on different
  grids or resolutions without any alignment assumption.
- Resampling is `average`, not `bilinear`. This is a stock quantity, so the resampling has to
  preserve the area weighted mean when the raster is reprojected to the reference CRS.
- Pool shares are a percentage of the biomass total reported here, not of total site carbon.
  With soil excluded, a share of the total is not a share of the site.

**Example render.**

> **1,284,000 tCO2e**
>
> This project area currently stores approximately 1,284,000 tCO2e in aboveground and
> belowground biomass, an average of 1,036 tCO2e per hectare. Aboveground biomass holds
> 1,027,200 tCO2e (80%) and belowground biomass 256,800 tCO2e (20%). Soil organic carbon is not
> included.

| Carbon pool | Storage (tCO2e) | Share of biomass carbon |
|---|---|---|
| Aboveground biomass | 1,027,200 | 80% |
| Belowground biomass | 256,800 | 20% |
| **Total** | **1,284,000** | **100%** |

**Narrative not yet specified.** The wording above is a placeholder written to make the units
and the scope explicit. Replace it once the team settles the phrasing.

**Downstream use.** The current stock is the reference point for Benefit Quantification in
F02-P5: avoided loss is measured against what is standing, and removals are measured as growth
towards a reference stock. The pool split matters there because the two pools behave differently
under disturbance: aboveground biomass is lost quickly in a clearing event, while root carbon
decays over years.

In [ ]:
CARBON_POOLS = ("Aboveground biomass", "Belowground biomass")


@dataclass(frozen=True)
class CarbonPool:
    """One biomass pool integrated over the AOI."""

    name: str
    biomass_mg: float      # total dry biomass, tonnes
    storage_tco2e: float   # after carbon fraction and 44/12
    coverage_pct: float    # share of the AOI with a valid pixel
    pct: float = 0.0       # share of the biomass carbon total, filled in once both pools exist


def _integrate_pool(name: str, path: str, aoi: AOI) -> CarbonPool:
    """Integrate one biomass raster over the AOI and convert to tCO2e.

    Density times area, so the pixel area cancels the per hectare unit:
        Mg/ha * ha = Mg
    Each pool is integrated on its own grid, so AGB and BGB need not share a resolution.
    """
    raster = load_raster_clipped(path, aoi, resampling="average")

    # Team decision: nodata inside the AOI counts as zero biomass. Coverage is measured
    # separately so an incomplete raster is still visible.
    values = raster.values.filled(0.0).astype(float)

    biomass_mg = float(values.sum()) * raster.pixel_area_ha
    storage_tco2e = biomass_mg * CARBON_FRACTION * CO2_PER_C

    return CarbonPool(
        name=name,
        biomass_mg=biomass_mg,
        storage_tco2e=storage_tco2e,
        coverage_pct=safe_pct(raster.valid_area_ha, aoi.area_ha),
    )


def analyze_current_carbon_storage(aoi: AOI) -> ComponentResult:
    """Component 3.1. Biomass carbon currently stored in the project area, in tCO2e."""
    pools = [
        _integrate_pool(CARBON_POOLS[0], AGB_RASTER, aoi),
        _integrate_pool(CARBON_POOLS[1], BGB_RASTER, aoi),
    ]

    total_tco2e = sum(p.storage_tco2e for p in pools)

    if total_tco2e <= 0:
        return not_applicable(
            "3.1 Current Carbon Storage",
            "No biomass data is available for this project area, so current carbon storage "
            "cannot be estimated.",
        )

    # Shares are of the biomass total reported here, not of total site carbon. Soil is excluded,
    # so these percentages sum to 100 of a partial accounting.
    pools = [replace(p, pct=safe_pct(p.storage_tco2e, total_tco2e)) for p in pools]

    density_tco2e_ha = total_tco2e / aoi.area_ha if aoi.area_ha > 0 else 0.0
    coverage_pct = max(p.coverage_pct for p in pools)

    flags: list[str] = []
    if coverage_pct < CARBON_COVERAGE_WARN_PCT:
        flags.append(
            f"3.1: the biomass rasters cover only {coverage_pct:.0f}% of the AOI. Nodata is "
            "counted as zero biomass, so the headline is an under-estimate by an unknown "
            "amount."
        )

    # Placeholder wording, see the note above.
    breakdown = oxford_join(
        f"{p.name.lower()} holds {p.storage_tco2e:,.0f} tCO2e ({fmt_pct(p.pct)})"
        for p in pools
    )
    narrative = (
        f"This project area currently stores approximately {total_tco2e:,.0f} tCO2e in "
        f"aboveground and belowground biomass, an average of {density_tco2e_ha:,.0f} tCO2e per "
        f"hectare. Of this, {breakdown}. Soil organic carbon is not included."
    )

    return ComponentResult(
        component="3.1 Current Carbon Storage",
        applicable=True,
        narrative=narrative,
        tables={"pools": pools},  # name, storage_tco2e, pct -> breakdown table
        values={
            "total_tco2e": total_tco2e,          # headline big number
            "density_tco2e_ha": density_tco2e_ha,
            "coverage_pct": coverage_pct,
            "pool_tco2e": {p.name: p.storage_tco2e for p in pools},
            "pool_pct": {p.name: p.pct for p in pools},
            "pools_included": list(CARBON_POOLS),
            "pools_excluded": ["deadwood", "litter", "soil organic carbon"],
        },
        flags=flags,
    )

---
## Run and save

In [ ]:
results: dict[str, ComponentResult] = {}

results["3.1"] = analyze_current_carbon_storage(aoi)

for key, r in results.items():
    print(f"[{key}] {r.component}{'' if r.applicable else '  (not applicable)'}")
    print(f"      {r.narrative}")
    for f in r.flags:
        print(f"      FLAG: {f}")

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_CLIMATE)
print(f"Saved {path}")